# Домашка 5. LLM и RAG

В этой домашке вам предстоит сделать чат-бота-врача, используя технику RAG (Retrieval-Augmented-Generation) и фреймворки huggingface и LangChain

In [ ]:
!pip install datasets langchain_community langchain_chroma langchain langchain_core tiktoken sentence-transformers==2.2.2 lark InstructorEmbedding bitsandbytes accelerate >> /dev/null

Загрузим датасет medal https://huggingface.co/datasets/bigbio/medal

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


### Задание 0. Загрузите данные MEDAL (0.5 балла)
Данные содержат медицинские статьи для различных клинических диагнозов

https://github.com/McGill-NLP/medal


Нас интересуют колонки TEXT и LABEL


In [ ]:
# https://zenodo.org/record/4482922/files/train.csv
# https://drive.google.com/file/d/1X7PTIkmsFhTk5n-4W6SWa7XWsDGTpafl/view?usp=sharing
!cat gdrive/MyDrive/HSE/medal_train.csv | head

ABSTRACT_ID,TEXT,LOCATION,LABEL
14145090,velvet antlers vas are commonly used in traditional chinese medicine and invigorant and contain many PET components for health promotion the velvet antler peptide svap is one of active components in vas based on structural study the svap interacts with tgfÎ² receptors and disrupts the tgfÎ² pathway we hypothesized that svap prevents cardiac fibrosis from pressure overload by blocking tgfÎ² signaling SDRs underwent TAC tac or a sham operation T3 one month rats received either svap mgkgday or vehicle for an additional one month tac surgery induced significant cardiac dysfunction FB activation and fibrosis these effects were improved by treatment with svap in the heart tissue tac remarkably increased the expression of tgfÎ² and connective tissue growth factor ctgf ROS species C2 and the phosphorylation C2 of smad and ERK kinases erk svap inhibited the increases in reactive oxygen species C2 ctgf expression and the phosphorylation of smad and erk bu

### Задание 1. Чтение и индексация данных (2.5 балла)

In [ ]:
from langchain_community.document_loaders.csv_loader import CSVLoader


FILE_PATH = ?
docs = ???
# Подсказка: Используйте .lazy_load() вместо .load() (первый возвращает генератор)

In [ ]:
from langchain.embeddings import HuggingFaceInstructEmbeddings
import torch

# задайте модель для эмбединга ваших документов в индексе
emb_model = ???

In [ ]:
from langchain.vectorstores import Chroma

persist_directory = 'DB'

# Задайте индекс для ваших эмбедингов
vectordb = ???
vectordb.persist()

In [ ]:
from tqdm.auto import tqdm

N_DOCS=2000
# Проиндексируйте документы. Можно первые N_DOCS штук (все 3млн долго)

for i,doc in tqdm(enumerate(docs), total=N_DOCS):
  ???

vectordb.persist()

In [ ]:
!ls -lht DB

In [ ]:
from langchain_community.llms.huggingface_pipeline import HuggingFacePipeline
from langchain.chains.query_constructor.base import AttributeInfo
from langchain.retrievers.self_query.base import SelfQueryRetriever
import torch

# Название модели
model_id = ???

# Загрузка модели в базовый класс Langchain LLM
llm = ???

retriever = vectordb.as_retriever() # Retriever - это класс-обёртка над индексом, который можно добавлять в chain для поиска документов в индексе

In [ ]:
retriever.invoke("ceftobiprole bpr") # проверим, что поиск по индексу работает

### Задание 2. Prompt Engineering. Создание Prompt Template (3 балла)

In [ ]:
# Посмотрим, как выглядит шаблон токенизатора вашей модели. Синтаксис jinja2 достаточно понятный
from pprint import pp as pprint
from transformers import AutoTokenizer
tokenizer = ???
pprint(tokenizer.chat_template)

In [ ]:
# PromptTemplate(template=tokenizer.chat_template, template_format='jinja2', input_variables=['content'])

Советы:

prompt template можно посмотреть на https://github.com/chujiezheng/chat_templates и replicate.com. Например, для LLama 3 они тут

https://replicate.com/meta/meta-llama-3-70b-instruct
https://github.com/chujiezheng/chat_templates/blob/main/chat_templates/llama-3-chat.jinja

In [ ]:
"""LLama 3 template:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a helpful assistant<|eot_id|><|start_header_id|>user<|end_header_id|>

{prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""

from langchain.prompts import PromptTemplate


SYSTEM_PROMPT = """Здесь ваш системный промпт"""


USE_HISTORY = False
if USE_HISTORY:
    # Задание под бонусом, см. конец ноутбука
    instruction = """
    Здесь ваша инструкция
    """
    prompt_template =
    prompt = PromptTemplate(input_variables=???, template=prompt_template)
else:
    instruction = """
    Здесь ваша инструкция
    """
    prompt_template =
    prompt = PromptTemplate(input_variables=???, template=prompt_template)
prompt

### Задание 3. Создание Chain (Langchain pipeline) (2 балла)

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers.string import StrOutputParser

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

chain = (
    {??? feature engineering stage} | # feature engineering (retrieval augmentation)
    {??? preprocessing stage} | # препроцессинг (prompt engineering)
    llm | # модель
    (??? postprocessing stage) # постпроцессинг
)

In [ ]:
chain.invoke('How to treat pneumonia?')

In [ ]:
chain.invoke('Tell in details what is ceftobiprole bpr?')

### Бонус (2 балла). Добавьте в пайплайн историю переписки с ботом
Подсказка: langchain.memory.ConversationBufferMemory